## Werkstudent Job Classifier

Classify scraped job postings with a local Ollama model.  
Each posting is labelled **`working_student`**, **`not_working_student`**, or **`unsure`**.  
When a skills profile is provided, a **`skills_matching`** percentage (0–100) is also appended.

**Run order:** execute cells top to bottom. Only **Cell 1 – Configuration** needs editing.

In [1]:
# ── Cell 2 · Imports ──────────────────────────────────────────────────────────
import json
import os
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

## Config

In [ ]:
# ── Cell 1 · Configuration ────────────────────────────────────────────────────
# get cwd
cwd = os.getcwd()

# Ollama connection
OLLAMA_BASE_URL = "http://localhost:11434"   # change if Ollama runs on a remote host
MODEL_NAME      = "gemma4_12b_q5:latest"    # exact name shown by `ollama list`
MAX_WORKERS = 6 # parralel running instances of classification llm
# requires $env:OLLAMA_NUM_PARALLEL = "6" aas gloabal powershell command

# Input / output
CSV_PATH = f"{cwd}/jobs.csv"   # path to your scraped jobs CSV
OUT_PATH = None         # None → auto-named  <csv_stem>_classified.csv  and  _final.csv

# Skills profile  (enables 0-100 % skills_matching column)
# Accepted CSV shape  (any columns, one row per skill):

SKILLS = f"{cwd}/skill_profile.csv" #None

# Set to an integer (e.g. 10) to process only the first N rows — useful for quick tests
LIMIT = 10000000000000000000000000000000 #None

# ── Cell 3 · Derived URLs (do not edit) ──────────────────────────────────────
CHAT_URL = f"{OLLAMA_BASE_URL}/api/chat"
TAGS_URL = f"{OLLAMA_BASE_URL}/api/tags"

VALID_CLASSIFICATIONS = {"working_student", "not_working_student", "unsure"}

<>:11: SyntaxWarning: invalid escape sequence '\j'
<>:17: SyntaxWarning: invalid escape sequence '\s'
<>:11: SyntaxWarning: invalid escape sequence '\j'
<>:17: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Jojo\AppData\Local\Temp\ipykernel_36964\348525156.py:11: SyntaxWarning: invalid escape sequence '\j'
  CSV_PATH = f"{cwd}\jobs.csv"   # path to your scraped jobs CSV
C:\Users\Jojo\AppData\Local\Temp\ipykernel_36964\348525156.py:17: SyntaxWarning: invalid escape sequence '\s'
  SKILLS = f"{cwd}\skill_profile.csv" #None


## Classification Prompt

In [3]:
# ── Cell 4 · System prompts ───────────────────────────────────────────────────

CLASSIFY_SYSTEM_PROMPT = """You are a strict job-posting classifier specialized in German "Werkstudent" \
(working student) roles. You will be given a job title and description. Decide \
whether this is a GENUINE working student position using the rubric below.

A genuine Werkstudent / working student role typically has ALL of these:
- Part-time hours, commonly stated as 10-20 hours per week (sometimes phrased as \
"bis zu 20 Stunden", "10-20h/Woche", "part-time alongside your studies", etc.)
- Requires the applicant to be CURRENTLY ENROLLED as a student -- at a university \
(Universitat), university of applied sciences (Hochschule/FH), in a Bachelor's, \
Master's program, or sometimes still in secondary/high school (Gymnasium) working \
toward university entry. Look for phrases like "eingeschriebener Student", \
"immatrikuliert", "laufendes Studium", "currently enrolled".
- The field of study is typically relevant to the role -- for data/tech roles, \
this usually means Mathematik, Data Science, Informatik (Computer Science), \
Wirtschaftsinformatik, Statistik, or closely related fields, though exact-major \
matching is not required if the role itself is reasonably technical.
- Often pays hourly rather than an annual salary, and is explicitly framed as \
flexible around lecture/exam periods (e.g. "flexibel rund um dein Studium").

Mark as "not_working_student" any of the following, even if the title contains \
the word "Werkstudent":
- Full-time positions (Vollzeit, 35-40+ hours/week, no part-time framing)
- Internships / Praktikum / Praktikant roles -- these are a DIFFERENT category \
from Werkstudent in Germany even though job boards often conflate them. If the \
description says "Praktikum", "Praktikant", "internship", or describes a fixed \
3-6 month full-time placement with no ongoing part-time structure, mark it \
"not_working_student", even if "Werkstudent" also appears somewhere in the text.
- Regular full-time entry-level, graduate, junior, or experienced-professional roles
- Roles with no part-time/student framing at all

If the description does not give enough detail to be sure either way -- e.g. no \
hours stated, no mention of student status, vague or truncated text -- classify \
it as "unsure" rather than guessing.

Respond with ONLY a JSON object, no other text, in this exact shape:
{"classification": "working_student" or "not_working_student" or "unsure", \
"reason": "one short sentence explaining your verdict, citing the specific phrase \
that drove the decision if possible"}
"""

CLASSIFY_USER_TEMPLATE = """Title: {title}
Company: {company}
Job type (as listed by the site, may be missing or wrong): {job_type}
Description:
{description}
"""

SKILLS_SYSTEM_PROMPT = CLASSIFY_SYSTEM_PROMPT + """

In ADDITION to the working-student classification above, you will also be given \
the candidate's skills and background. After classifying the role, judge how good \
a fit this specific job is for this specific candidate, based on the description's \
required/preferred skills, the field of study mentioned, and the seniority implied.

Score the fit as "skills_matching": an integer PERCENTAGE from 0 to 100, where \
0 means no overlap at all between the candidate's skills and the job's requirements, \
and 100 means an excellent, near-complete match. Use the full range -- do not \
default to round numbers like 50 unless that genuinely reflects a partial match.

Respond with ONLY a JSON object, no other text, in this exact shape:
{"classification": "working_student" or "not_working_student" or "unsure", \
"reason": "one short sentence on the working-student verdict", \
"skills_matching": an integer from 0 to 100, \
"skills_matching_reason": "one to two short sentences on why this is or isn't a \
good fit for this specific candidate, mentioning the most relevant matching or \
missing skills"}
"""

SKILLS_USER_TEMPLATE = """Candidate skills and background:
{skills}

Job posting:
Title: {title}
Company: {company}
Job type (as listed by the site, may be missing or wrong): {job_type}
Description:
{description}
"""

## Helper Functions

In [4]:
# ── Cell 5 · Core helpers ─────────────────────────────────────────────────────
import urllib

def get_available_models():
    try:
        with urllib.request.urlopen(TAGS_URL, timeout=10) as resp:
            data = json.loads(resp.read().decode("utf-8"))
        return [m["name"] for m in data.get("models", [])], None
    except urllib.error.URLError as e:
        return None, str(e.reason)
    except Exception as e:
        return None, f"{type(e).__name__}: {e}"


def call_ollama(system_prompt, user_prompt, timeout=90, retries=2):
    """POST to Ollama chat endpoint; return dict with _ok, _error, _raw keys."""
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        "format":  "json",
        "stream":  False,
        "options": {"temperature": 0.0},
    }
    data = json.dumps(payload).encode("utf-8")
    req  = urllib.request.Request(
        CHAT_URL, data=data,
        headers={"Content-Type": "application/json"}, method="POST"
    )
    last_error = None
    for attempt in range(retries + 1):
        try:
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                body = json.loads(resp.read().decode("utf-8"))
            content = body.get("message", {}).get("content", "")
            return {"_ok": True, "_error": None, "_raw": json.loads(content)}
        except urllib.error.HTTPError as e:
            err_body = e.read().decode("utf-8", errors="replace")
            if e.code == 404:
                last_error = (
                    f"Model '{MODEL_NAME}' not found (HTTP 404). "
                    f"Run 'ollama list' and copy the exact name. Server: {err_body}"
                )
            else:
                last_error = f"Ollama HTTP {e.code}: {err_body}"
            break
        except urllib.error.URLError as e:
            last_error = f"Cannot reach Ollama at {CHAT_URL} — is 'ollama serve' running? ({e})"
            break
        except (json.JSONDecodeError, KeyError, TypeError) as e:
            last_error = f"Unparseable model output (attempt {attempt + 1}): {e}"
            time.sleep(1)
        except Exception as e:
            last_error = f"Unexpected error (attempt {attempt + 1}): {type(e).__name__}: {e}"
            time.sleep(1)
    return {"_ok": False, "_error": last_error, "_raw": None}


def _normalize_classification(value):
    v = str(value or "").strip().lower()
    if v in VALID_CLASSIFICATIONS:
        return v
    aliases = {
        "working student":         "working_student",
        "yes": "working_student",  "true": "working_student",
        "not working student":     "not_working_student",
        "not_a_working_student":   "not_working_student",
        "no": "not_working_student", "false": "not_working_student",
        "uncertain": "unsure",     "unclear": "unsure", "unknown": "unsure",
    }
    return aliases.get(v, "unsure")


def classify_row(title, company, job_type, description):
    user_prompt = CLASSIFY_USER_TEMPLATE.format(
        title=title or "(missing)",
        company=company or "(missing)",
        job_type=job_type or "(missing)",
        description=(description or "(no description available)")[:4000],
    )
    result = call_ollama(CLASSIFY_SYSTEM_PROMPT, user_prompt)
    if not result["_ok"]:
        return {"classification": None,
                "reason": "Classification failed: " + str(result["_error"])}
    p = result["_raw"]
    return {"classification": _normalize_classification(p.get("classification")),
            "reason": str(p.get("reason", "")).strip()}


def classify_row_with_fit(title, company, job_type, description, skills):
    user_prompt = SKILLS_USER_TEMPLATE.format(
        skills=skills,
        title=title or "(missing)",
        company=company or "(missing)",
        job_type=job_type or "(missing)",
        description=(description or "(no description available)")[:4000],
    )
    result = call_ollama(SKILLS_SYSTEM_PROMPT, user_prompt)
    if not result["_ok"]:
        return {"classification": None,
                "reason": "Classification failed: " + str(result["_error"]),
                "skills_matching": None, "skills_matching_reason": ""}
    p = result["_raw"]
    sm = p.get("skills_matching")
    try:
        sm = max(0, min(100, int(round(float(sm))))) if sm is not None else None
    except (TypeError, ValueError):
        sm = None
    return {"classification":         _normalize_classification(p.get("classification")),
            "reason":                 str(p.get("reason", "")).strip(),
            "skills_matching":        sm,
            "skills_matching_reason": str(p.get("skills_matching_reason", "")).strip()}

In [5]:
# ── Cell 6 · Skills profile loader ───────────────────────────────────────────

def _fmt_json(data):
    lines = []
    if isinstance(data, list):
        for item in data:
            if isinstance(item, dict):
                skill   = item.get("skill") or item.get("name") or json.dumps(item)
                extras  = {k: v for k, v in item.items() if k not in ("skill", "name")}
                extra_s = (" (" + ", ".join(f"{k}: {v}" for k, v in extras.items()) + ")") if extras else ""
                lines.append("- " + str(skill) + extra_s)
            else:
                lines.append("- " + str(item))
    elif isinstance(data, dict):
        for key, value in data.items():
            lines.append(str(key) + ": " + (", ".join(str(v) for v in value) if isinstance(value, list) else str(value)))
    else:
        lines.append(str(data))
    return "\n".join(lines)


def _fmt_csv(df):
    cols  = list(df.columns)
    lines = []
    for _, row in df.iterrows():
        parts = [f"{c}: {row[c]}" for c in cols if pd.notna(row[c])]
        if parts:
            lines.append("- " + ", ".join(parts))
    return "\n".join(lines)


def load_skills_profile(skills_arg):
    """
    Accept inline text OR a .json / .csv file path and return a single text
    block ready to drop into the LLM prompt. Falls back to plain text if the
    path doesn't exist on disk.
    """
    if skills_arg is None:
        return None
    candidate_path = skills_arg.strip()
    lower = candidate_path.lower()

    if lower.endswith(".json") and os.path.isfile(candidate_path):
        with open(candidate_path, "r", encoding="utf-8") as f:
            return _fmt_json(json.load(f))

    if lower.endswith(".csv") and os.path.isfile(candidate_path):
        return _fmt_csv(pd.read_csv(candidate_path))

    if (lower.endswith(".json") or lower.endswith(".csv")) and not os.path.isfile(candidate_path):
        print(f"WARNING: '{candidate_path}' looks like a file path but was not found — "
              "treating SKILLS as plain text.")

    return skills_arg

In [6]:
# ── Cell 7 · Validate Ollama connection and model ────────────────────────────

available = get_available_models()
if available is None:
    raise RuntimeError(
        f"Cannot reach Ollama at {OLLAMA_BASE_URL}.\n"
        "Make sure 'ollama serve' is running, and that OLLAMA_BASE_URL is correct."
    )

available, conn_error = get_available_models()

if available is None:
    raise RuntimeError(
        f"Cannot reach Ollama at {OLLAMA_BASE_URL}.\n"
        f"Connection error: {conn_error}\n\n"
        "The Windows error 'Normalerweise darf jede Socketadresse ... nur jeweils einmal verwendet werden'\n"
        "means Ollama IS already running — do NOT run 'ollama serve' again, just re-run this cell.\n"
        "If Ollama truly isn't running, open a new terminal and run: ollama serve\n"
        "If it's on a different host/port, update OLLAMA_BASE_URL in Cell 1."
    )

print(f"✓ Ollama reachable at {OLLAMA_BASE_URL}")
print(f"✓ Model '{MODEL_NAME}' found")

✓ Ollama reachable at http://localhost:11434
✓ Model 'gemma4_12b_q5:latest' found


## Load CSV

In [7]:
# ── Cell 8 · Load CSV and detect columns ──────────────────────────────────────

df = pd.read_csv(CSV_PATH)
if LIMIT:
    df = df.head(LIMIT).copy()
    print(f"LIMIT set — using first {LIMIT} rows only.")

title_col   = next((c for c in ["title",       "job_title"]       if c in df.columns), None)
desc_col    = next((c for c in ["description", "job_description"] if c in df.columns), None)
company_col = next((c for c in ["company",     "company_name"]    if c in df.columns), None)
job_type_col= next((c for c in ["job_type"]                       if c in df.columns), None)

if title_col is None:
    raise ValueError(f"No title column found. Columns present: {list(df.columns)}")
if desc_col is None:
    print("WARNING: no description column found — accuracy will be much lower.")

skills_profile = load_skills_profile(SKILLS)
fit_mode       = skills_profile is not None

print(f"Loaded {len(df)} rows from '{CSV_PATH}'")
print(f"Columns detected — title: '{title_col}' | description: '{desc_col}' | "
      f"company: '{company_col}' | job_type: '{job_type_col}'")
if fit_mode:
    print("\nSkills-matching mode ON. Profile:\n" + skills_profile)

LIMIT set — using first 10 rows only.
Loaded 10 rows from 'd:\Projects\Coding\Get_Job_Offerings\jobspy\jobs.csv'
Columns detected — title: 'title' | description: 'description' | company: 'company' | job_type: 'None'

Skills-matching mode ON. Profile:
- name: python,  skill_lvl: 70%,  area: python
- name: numpy,  skill_lvl: 75%,  area: python
- name: pandas,  skill_lvl: 90%,  area: python
- name: mcp-server,  skill_lvl: 68%,  area: python
- name: matplotlib,  skill_lvl: 90%,  area: python
- name: tensorflow,  skill_lvl: 20%,  area: python
- name: pytorch,  skill_lvl: 70%,  area: python 
- name: glue,  skill_lvl: 80%,  area: aws
- name: quick,  skill_lvl: 100%,  area: aws
- name: hadoop/pyspark,  skill_lvl: 50%,  area: aws
- name: dataoptimization,  skill_lvl: 75%,  area: aws
- name: datagovernance,  skill_lvl: 83%,  area: aws
- name: powershell,  skill_lvl: 40%,  area: shell
- name: api,  skill_lvl: 40%,  area: shell
- name: mysql,  skill_lvl: 65%,  area: sql
- name: postgres-sql,  skil

## Classification

In [8]:
# ── Cell 9 · Run classification (parallel) ────────────────────────────────────
import threading
_print_lock = threading.Lock()

def _classify_one(args):
    idx, row = args
    title       = str(row.get(title_col,    "")) if title_col    else ""
    company     = str(row.get(company_col,  "")) if company_col  else ""
    job_type    = str(row.get(job_type_col, "")) if job_type_col else ""
    description = (
        str(row.get(desc_col, ""))[:1500]          # 4000 → 1500: cuts tokens, same accuracy
        if desc_col and pd.notna(row.get(desc_col))
        else ""
    )
    if fit_mode:
        verdict = classify_row_with_fit(title, company, job_type, description, skills_profile)
        sm_str  = f", skills_matching={verdict['skills_matching']}%" if verdict["skills_matching"] is not None else ""
    else:
        verdict = classify_row(title, company, job_type, description)
        sm_str  = ""
    return idx, title, verdict, sm_str

results_map = {}
total = len(df)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(_classify_one, (i, row)): i for i, row in df.iterrows()}
    for n, future in enumerate(as_completed(futures), 1):
        idx, title, verdict, sm_str = future.result()
        results_map[idx] = verdict
        with _print_lock:
            print(f"[{n}/{total}] {title[:60]!r} → {verdict['classification']}{sm_str}")

# Reassemble in original CSV order
results = [results_map[i] for i in df.index]
print("\n✓ Classification complete.")

[1/10] 'Werkstudent (m/w/d) Digital Analytics & Tracking Engineering' → unsure, skills_matching=65%
[2/10] 'Werkstudent AI & LLM Engineering (all genders) (all genders)' → working_student, skills_matching=85%
[3/10] 'Werkstudent/in Sparkassenprüfung/Wirtschaftsprüfung mit Schw' → working_student, skills_matching=85%
[4/10] 'Werkstudent Quality & Product Compliance Management (m/w/d)' → working_student, skills_matching=85%
[5/10] 'Werkstudent Künstliche Intelligenz & Digitalisierung (m/w/d)' → working_student, skills_matching=85%
[6/10] 'Praktikant / Werkstudent (m/w/d) KI-Assistent & Wissensquali' → working_student, skills_matching=85%
[7/10] 'Werkstudent AI Engineer (m/w/d)' → unsure, skills_matching=85%
[8/10] 'Werkstudent im Bereich agiles Projektmanagement - Innovation' → working_student, skills_matching=78%
[9/10] 'Werkstudent Data Visualization and Analysis (all genders)' → working_student, skills_matching=85%
[10/10] 'Werkstudent | IT & Development (m/w/d) | Coding, Automatisie'

## Save

In [ ]:
# ── Cell 10 · Append results and save full classified CSV ─────────────────────

df["classification"] = [r["classification"] for r in results]
df["match_reason"]   = [r["reason"]         for r in results]

if fit_mode:
    df["skills_matching"]        = [r["skills_matching"]        for r in results]
    df["skills_matching_reason"] = [r["skills_matching_reason"] for r in results]

stem      = OUT_PATH.rsplit(".", 1)[0] if OUT_PATH else CSV_PATH.rsplit(".", 1)[0]
full_path = stem + "_classified.csv"
df.to_csv(full_path, index=False)

n_ws    = (df["classification"] == "working_student").sum()
n_not   = (df["classification"] == "not_working_student").sum()
n_unsure= (df["classification"] == "unsure").sum()
n_err   = df["classification"].isna().sum()

print(f"Saved full results ({len(df)} rows) → {full_path}")
print(f"  working_student:     {n_ws}")
print(f"  not_working_student: {n_not}  ← will be dropped in final output")
print(f"  unsure:              {n_unsure}")
print(f"  errors:              {n_err}")
#df.head(4)

Saved full results (10 rows) → d:\Projects\Coding\Get_Job_Offerings\jobspy\jobs_classified.csv
  working_student:     8
  not_working_student: 0  ← will be dropped in final output
  unsure:              2
  errors:              0


,id,site,job_url,title,company,location,date_posted,emails,description,company_industry,company_url,classification,match_reason,skills_matching,skills_matching_reason
0,gd-1010185196213,glassdoor,https://www.glassdoor.de/job-listing/j?jl=1010...,Werkstudent Quality & Product Compliance Manag...,Knorr-Bremse,München,2026-07-01,NaN,**ARBEITSORT:** München / Deutschland | **UNTE...,NaN,https://www.glassdoor.de/Overview/W-EI_IE22136...,working_student,The role is explicitly titled 'Werkstudent' an...,85,The candidate has strong skills in Data Scienc...
1,gd-1010182609963,glassdoor,https://www.glassdoor.de/job-listing/j?jl=1010...,Werkstudent Künstliche Intelligenz & Digitalis...,Bayern Facility Management,München,2026-06-30,bewerbung@bayernfm.de,**Werden Sie Teil unseres Teams.**\n\n#### **I...,NaN,https://www.glassdoor.de/Overview/W-EI_IE13353...,working_student,The job description explicitly requires an 'Ei...,85,The candidate has strong skills in Data Scienc...
2,gd-1010179417154,glassdoor,https://www.glassdoor.de/job-listing/j?jl=1010...,Werkstudent AI & LLM Engineering (all genders)...,EXXETA,München,2026-06-26,NaN,München\n\nBei Exxeta fordern wir das traditio...,NaN,https://www.glassdoor.de/Overview/W-EI_IE40696...,working_student,The job description explicitly states 'Du stud...,85,The candidate has strong skills in Python (70%...
3,gd-1010178590097,glassdoor,https://www.glassdoor.de/job-listing/j?jl=1010...,Werkstudent/in Sparkassenprüfung/Wirtschaftspr...,Sparkassenverband Bayern,München,2026-06-25,NaN,Dein Karrierestart im Herzen Münchens bei der ...,NaN,https://www.glassdoor.de/Overview/W-EI_IE91574...,working_student,The job title and description explicitly state...,85,The candidate has a very strong match in Pytho...


In [10]:
# ── Cell 11 · Filter and save final CSV ───────────────────────────────────────
# Drops not_working_student rows; keeps working_student + unsure.
# When skills-matching is on, sorts best matches first.

final_df = df[df["classification"].isin(["working_student", "unsure"])].copy()

if fit_mode and not final_df.empty:
    final_df = final_df.sort_values("skills_matching", ascending=False)

final_path = stem + "_final.csv"
final_df.to_csv(final_path, index=False)

sort_note = " (sorted by skills_matching, best first)" if fit_mode else ""
print(f"Saved final results ({len(final_df)} rows) → {final_path}{sort_note}")
print(f"  not_working_student rows dropped: {n_not}")

final_df

Saved final results (10 rows) → d:\Projects\Coding\Get_Job_Offerings\jobspy\jobs_final.csv (sorted by skills_matching, best first)
  not_working_student rows dropped: 0


,id,site,job_url,title,company,location,date_posted,emails,description,company_industry,company_url,classification,match_reason,skills_matching,skills_matching_reason
9,gd-1010170337459,glassdoor,https://www.glassdoor.de/job-listing/j?jl=1010...,Werkstudent | IT & Development (m/w/d) | Codin...,DARKSIDE,München,2026-06-17,karriere@rocksolid-personal.de,#### **Deine Aufgaben**\n\nJe nach Projekt arb...,NaN,https://www.glassdoor.de/Overview/W-EI_IE84170...,working_student,The job explicitly states '15-20 Stunden/Woche...,88,"The candidate has strong skills in Python, SQL..."
0,gd-1010185196213,glassdoor,https://www.glassdoor.de/job-listing/j?jl=1010...,Werkstudent Quality & Product Compliance Manag...,Knorr-Bremse,München,2026-07-01,NaN,**ARBEITSORT:** München / Deutschland | **UNTE...,NaN,https://www.glassdoor.de/Overview/W-EI_IE22136...,working_student,The role is explicitly titled 'Werkstudent' an...,85,The candidate has strong skills in Data Scienc...
1,gd-1010182609963,glassdoor,https://www.glassdoor.de/job-listing/j?jl=1010...,Werkstudent Künstliche Intelligenz & Digitalis...,Bayern Facility Management,München,2026-06-30,bewerbung@bayernfm.de,**Werden Sie Teil unseres Teams.**\n\n#### **I...,NaN,https://www.glassdoor.de/Overview/W-EI_IE13353...,working_student,The job description explicitly requires an 'Ei...,85,The candidate has strong skills in Data Scienc...
2,gd-1010179417154,glassdoor,https://www.glassdoor.de/job-listing/j?jl=1010...,Werkstudent AI & LLM Engineering (all genders)...,EXXETA,München,2026-06-26,NaN,München\n\nBei Exxeta fordern wir das traditio...,NaN,https://www.glassdoor.de/Overview/W-EI_IE40696...,working_student,The job description explicitly states 'Du stud...,85,The candidate has strong skills in Python (70%...
3,gd-1010178590097,glassdoor,https://www.glassdoor.de/job-listing/j?jl=1010...,Werkstudent/in Sparkassenprüfung/Wirtschaftspr...,Sparkassenverband Bayern,München,2026-06-25,NaN,Dein Karrierestart im Herzen Münchens bei der ...,NaN,https://www.glassdoor.de/Overview/W-EI_IE91574...,working_student,The job title and description explicitly state...,85,The candidate has a very strong match in Pytho...
5,gd-1010172466536,glassdoor,https://www.glassdoor.de/job-listing/j?jl=1010...,Praktikant / Werkstudent (m/w/d) KI-Assistent ...,CHECK24,München,2026-06-19,anja.herger@check24.de,**Im Überblick**\n----------------\n\nStudent\...,NaN,https://www.glassdoor.de/Overview/W-EI_IE94874...,working_student,The job description explicitly mentions 'Laufe...,85,"The candidate has strong skills in Python, Dat..."
6,gd-1010173511524,glassdoor,https://www.glassdoor.de/job-listing/j?jl=1010...,Werkstudent AI Engineer (m/w/d),Estateanfrage,München,2026-06-19,NaN,NaN,NaN,https://www.glassdoor.de/Overview/W-EI_IE10757...,unsure,"The job title contains 'Werkstudent', but ther...",85,"The candidate has strong skills in Python, Pan..."
8,gd-1010169940957,glassdoor,https://www.glassdoor.de/job-listing/j?jl=1010...,Werkstudent Data Visualization and Analysis (a...,BNP Paribas,München,2026-06-17,NaN,In München suchen wir dich als Werkstudent Wer...,NaN,https://www.glassdoor.de/Overview/W-EI_IE10342...,working_student,The job title explicitly states 'Werkstudent' ...,85,"The candidate has strong skills in Python, Pan..."
7,gd-1010171210018,glassdoor,https://www.glassdoor.de/job-listing/j?jl=1010...,Werkstudent im Bereich agiles Projektmanagemen...,Allianz,München,2026-06-18,recruiting-operations@allianz.de,Willkommen bei der Allianz\n \n \nProjektman...,NaN,https://www.glassdoor.de/Overview/W-EI_IE3062.htm,working_student,The job is explicitly titled 'Werkstudent' and...,78,The candidate has strong data science skills (...
4,gd-1010173788155,glassdoor,https://www.glassdoor.de/job-listing/j?jl=1010...,Werkstudent (m/w/d) Digital Analytics & Tracki...,FELD M GmbH,München,2026-06-20,NaN,NaN,NaN,https://www.glassdoor.de/Overview/W-EI_IE31583...,unsure,"The job title contains 'Werkstudent', but ther...",65,"The candidate has stro